
<img src="img/viu_logo.png" width="200">

## 01MIAR - Generadores de Datos

![logo](img/python_logo.png)

*Ivan Fuertes / Franklin Alvarez*

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Como leer este notebook (version mejorada)</b>

Este notebook fue **enriquecido** con celdas adicionales de teoria y ejercicios. Vas a encontrar tres tipos de bloques de colores:

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AZUL = TEORIA</b></div> -> explicacion en palabras simples antes de cada concepto.

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AMARILLO = TU TURNO</b></div> -> el ejercicio que TU debes resolver. Hay una celda vacia debajo donde escribes el codigo.

<div style="background-color:#ebfbee;border-left:6px solid #2f9e44;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>VERDE = SOLUCION</b></div> -> codigo de referencia comentado por si te trabas.

> **Promesa importante**: el codigo y la teoria del profesor original no se han tocado. Solo se han ANADIDO bloques.

</div>

## Generadores en Python
- Yield
- Iterador Lazy

- Exercises (https://www.w3resource.com/python-exercises/generators-yield/index.php)

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Iterables, iteradores y generadores: la jerarquia</b>

Antes de entender un **generador** hay que dominar tres conceptos que se confunden mucho.

#### 1. Iterable

Un objeto **iterable** es **algo por lo que se puede iterar** (recorrer con `for`). Implementa el metodo magico `__iter__()` que devuelve un iterador.

Ejemplos: `list`, `tuple`, `str`, `dict`, `set`, `range`, los ficheros abiertos, los generadores.

```python
mi_lista = [1, 2, 3]
for x in mi_lista:    # funciona porque la lista es iterable
    print(x)
```

#### 2. Iterador (iterator)

Un **iterador** es el objeto que produce los valores **uno a uno** cuando le llamas `next(...)`. Implementa DOS metodos magicos:

- `__iter__()` -> se devuelve a si mismo.
- `__next__()` -> devuelve el siguiente valor o lanza `StopIteration`.

```python
mi_lista = [10, 20, 30]
it = iter(mi_lista)        # iter() llama a __iter__() y crea el iterador
print(next(it))             # 10
print(next(it))             # 20
print(next(it))             # 30
print(next(it))             # StopIteration -> error
```

| Concepto | Que es | Que metodos tiene |
|---|---|---|
| **Iterable** | algo recorrible | `__iter__` |
| **Iterator** | algo que da valores uno a uno | `__iter__` y `__next__` |

> Una lista es iterable PERO NO es iterador: necesitas pasarle por `iter(...)` primero. En cambio, un generador es **las dos cosas a la vez**.

#### 3. Generador (generator)

Un **generador** es una **forma facil de crear un iterador** sin tener que escribir clases con `__iter__` y `__next__` a mano.

Hay dos formas de crearlo:

- **Funcion generadora**: una funcion que usa `yield` en lugar de `return`.
- **Generator expression**: como una list comprehension pero con `( )` en lugar de `[ ]`.

#### Por que son utiles los generadores en IA?

| Problema | Solucion con generador |
|---|---|
| Un dataset no cabe en RAM (millones de imagenes, GB de texto) | Lees por **lotes** sin cargar todo |
| Procesar un fichero gigante linea a linea | Generator pipeline (lo veras abajo) |
| Data augmentation aleatoria | El generador produce variaciones infinitas |
| Streaming de datos en tiempo real | El generador NUNCA termina |

Keras y PyTorch dependen MUCHO de generadores para alimentar redes neuronales con datos que no caben en memoria.

</div>

<div style="background-color:#f3f0ff;border-left:6px solid #7048e8;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#4c2d9e;font-size:14px;">PARA RECORDAR</b>

**Evaluacion perezosa (lazy evaluation)**: el principio que hace especiales a los generadores.

Una lista calcula **TODOS** los valores y los guarda en memoria al instante.
Un generador NO calcula nada hasta que le pides el siguiente valor con `next(...)` o lo recorres con `for`.

```python
# 'Comprehension' INMEDIATA: ocupa RAM ahora mismo
cuadrados = [x**2 for x in range(1_000_000_000)]   # CRASH (8 GB)

# Generador PEREZOSO: solo calcula bajo demanda
cuadrados = (x**2 for x in range(1_000_000_000))   # OK, casi 0 bytes
```

El generador solo recuerda **donde se quedo**. Es como un libro con un marcapaginas: no lees las paginas hasta que las pides.

</div>

In [ ]:
# Devuelve lista
def squares_fun(n):
    result = []
    for i in range(n):
        result.append(i ** 2)
    return result

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Lista vs generador: comparativa real</b>

Aqui ves la **forma clasica** (devolver una lista). Compara despues con el generador (siguiente celda).

| Aspecto | `squares_fun(50000)` (lista) | `squares(50000)` (generador) |
|---|---|---|
| Memoria al crearlo | reserva 50000 enteros | reserva casi nada |
| Tiempo de creacion | calcula los 50000 | instantaneo |
| Reutilizable? | si, recorrible varias veces | **NO**, una sola pasada |
| Acceso por indice | si: `lista[100]` | NO: `gen[100]` -> error |
| len() | si | NO |
| Util para datos grandes | NO | SI |

#### Cuando usar uno u otro?

- **Lista**: si vas a recorrer los datos varias veces, si necesitas indexacion, si caben en RAM y son rapidos de calcular.
- **Generador**: si los datos son enormes, si solo los recorres una vez, si quieres componer un pipeline, si los produces de forma incremental (sensores, web scraping, etc.).

</div>

In [ ]:
lista = squares_fun(50000)
print(type(lista))
print(lista)
for i in lista:
    print(f"{i}")

In [ ]:
# Usar generador
def squares(n):
    for i in range(n):
        yield i ** 2

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; <code>yield</code>: la palabra magica que crea un generador</b>

Cualquier funcion que contenga `yield` (en cualquier parte del cuerpo) **deja de ser una funcion normal** y pasa a ser una **funcion generadora**.

```python
def squares(n):
    for i in range(n):
        yield i ** 2     # <- yield, no return
```

#### Que pasa cuando llamas a `squares(5)`?

**NO ejecuta el cuerpo**. Solo devuelve un objeto **generator**:

```python
gen = squares(5)
print(gen)               # <generator object squares at 0x...>
print(type(gen))         # <class 'generator'>
```

#### Cuando se ejecuta el cuerpo?

Solo cuando llamas a `next(gen)` (o iteras con `for`):

1. Primer `next(gen)`: ejecuta hasta el primer `yield`, devuelve el valor y **PAUSA** la funcion exactamente ahi.
2. Segundo `next(gen)`: **REANUDA** desde el `yield` anterior y vuelve a parar en el siguiente.
3. Cuando la funcion termina (sin mas `yield`): lanza `StopIteration`.

```
[next] -> ejecuta -> yield 0  (PAUSA)
[next] -> reanuda -> yield 1  (PAUSA)
[next] -> reanuda -> yield 4  (PAUSA)
...
[next] -> reanuda -> fin        -> StopIteration
```

> **Lo magico**: el generador **conserva su estado** (variables locales, posicion del bucle) entre llamadas. Es como si la funcion durmiera entre yields y se despertara con cada `next()`.

#### Diferencias con `return`

| `return` | `yield` |
|---|---|
| Termina la funcion | **Pausa** la funcion |
| Devuelve UN valor y se acabo | Puede devolver muchos, uno por llamada |
| El estado local se pierde | El estado local se preserva |

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 1</b>

Define una **funcion generadora** llamada `pares_hasta(n)` que produzca todos los **numeros pares desde 0 hasta n** (incluido si es par).

Despues:
1. Crea el generador `g = pares_hasta(10)`.
2. Comprueba su tipo con `type(g)`.
3. Recorrelo con un `for` e imprime cada valor.

> Resultado esperado: 0, 2, 4, 6, 8, 10.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# def pares_hasta(n):
#     # bucle desde 0 hasta n incluido
#     for i in range(n + 1):
#         # solo emite los que son pares
#         if i % 2 == 0:
#             yield i   # cada yield pausa la funcion
#
# # Llamar a la funcion crea el generador, NO ejecuta el cuerpo
# g = pares_hasta(10)
# print(type(g))   # <class 'generator'>
#
# # Iterar consume los valores uno a uno
# for x in g:
#     print(x)


In [ ]:
gen = squares(5)
# print(type(gen))
print(gen)
for i in gen:
    print(f"{i}")

- Next (envia None al generador)

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; <code>next()</code>: avanzar un paso en el generador</b>

`next(gen)` le dice al generador: **dame el siguiente valor**.

```python
def basico():
    print('A')
    yield 1
    print('B')
    yield 2
    print('C')
    yield 3

g = basico()         # NO imprime nada todavia (no se ha ejecutado el cuerpo)
print(next(g))       # imprime 'A' y luego 1
print(next(g))       # imprime 'B' y luego 2
print(next(g))       # imprime 'C' y luego 3
print(next(g))       # StopIteration -> excepcion
```

#### Patron tipico: `next()` con valor por defecto

Para evitar la excepcion `StopIteration` cuando se acaba el generador:

```python
valor = next(gen, None)        # devuelve None en lugar de fallar
valor = next(gen, 'sin datos')   # mensaje por defecto
```

#### Cuando usar `next()` directamente?

- Para **inspeccionar el primer elemento** sin recorrerlo todo.
- Para **avanzar manualmente** el generador (saltar la cabecera de un CSV, por ejemplo).
- En tests para verificar la salida del generador paso a paso.

En el codigo de arriba veras que cada `next(gen)` ejecuta el codigo hasta el siguiente `yield` y se queda esperando.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 2</b>

Define un generador `cuenta_atras(n)` que emita los numeros de `n` hasta `0` (incluidos).

Despues:
1. Crea `g = cuenta_atras(3)`.
2. Llama a `next(g)` cuatro veces e imprime cada resultado. La cuarta llamada deberia funcionar todavia (devuelve 0).
3. Llama una **quinta** vez con valor por defecto: `next(g, 'fin')`. Imprimelo.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# def cuenta_atras(n):
#     # bucle de n hasta 0, decrementando 1
#     while n >= 0:
#         yield n          # entrega n y pausa
#         n -= 1           # al reanudar, decrementa
#
# g = cuenta_atras(3)        # crea el generador (no ejecuta nada todavia)
# print(next(g))             # 3
# print(next(g))             # 2
# print(next(g))             # 1
# print(next(g))             # 0
# print(next(g, 'fin'))      # 'fin' (con default evita StopIteration)


In [ ]:
# Usar generador
def squares(n):
    for i in range(n):
        yield i ** 2
        print("hola")

def basico():
    print("Primera llamada")
    yield 1
    print("Segunda llamada")
    yield 2


In [ ]:
gen = basico()
print(type(gen))

In [ ]:
next(gen)

- Send (se pueden enviar valores en cada iteración)

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; <code>.send(valor)</code>: enviar datos AL generador</b>

`yield` no solo **devuelve** un valor, tambien puede **recibir** un valor del exterior.

```python
def gen():
    x = 0
    for i in range(10):
        x = yield i * x          # yield emite i*x Y recibe el valor que llegue por .send()
        print(f'recibi {x}')
```

#### Como funciona

1. La primera vez tienes que hacer `next(g)` (no `send`) para **arrancar** el generador.
2. Despues, `g.send(valor)` se comporta como `next()` PERO ademas: dentro del generador, la **expresion `yield`** se evalua al `valor` enviado.

```python
g = gen()
next(g)         # 0   (arranca, x todavia es 0, i=0, emite i*x = 0)
g.send(2)       # ahora yield = 2 -> x = 2; siguiente iteracion: i=1, emite 1*2 = 2
g.send(10)      # yield = 10 -> x = 10; i=2, emite 2*10 = 20
```

#### Para que sirve esto?

- **Corutinas** (programacion concurrente cooperativa).
- **Maquinas de estados** que reciben eventos.
- En `asyncio` y librerias asincronas, `send` es la base.

> Es un tema avanzado. En el dia a dia de IA usaras `yield` simple muchisimo mas que `.send()`.

</div>

In [ ]:
# Generator recibe datos
def gen():
    x = 0
    for i in range(10):
        x = yield i * x
        print(f"{x} inside function")

In [ ]:
g = gen()
print(type(g))

In [ ]:
next(g)

In [ ]:
a = g.send(2)
print(a)

## Expresiones Generadoras
- Comprehension List

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; List comprehension vs Generator expression</b>

La sintaxis es **casi identica**. La unica diferencia visible son los **parentesis**:

```python
[ x**2 for x in range(5) ]    # LIST comprehension  -> crea una lista [0,1,4,9,16]
( x**2 for x in range(5) )    # GENERATOR expression -> crea un generador perezoso
```

#### Comparativa

| Aspecto | `[x for x in ...]` lista | `(x for x in ...)` generador |
|---|---|---|
| Sintaxis | corchetes `[ ]` | parentesis `( )` |
| Resultado | `list` | `generator` |
| Memoria | toda en RAM al crear | casi 0 bytes |
| Velocidad de creacion | calcula todo ya | instantanea |
| Reutilizable | si | NO, un solo uso |
| Indexable | si: `lista[5]` | no |
| Tiene `len()` | si | no |
| `print()` muestra | los elementos | `<generator object ...>` |

#### Truco: parentesis se pueden omitir como argumento

Cuando el generator expression es el **unico argumento** a una funcion, no hace falta escribir los parentesis dobles:

```python
sum(x**2 for x in range(100))     # bonito
sum((x**2 for x in range(100)))  # tambien valido pero sobra un par
```

#### Casos perfectos para generator expression

```python
# Sumar valores que no caben en lista
total = sum(linea.count(',') for linea in open('huge.csv'))

# Encontrar primero que cumple condicion
primer_par = next(x for x in numeros if x % 2 == 0)

# Pipeline encadenado (lo veras abajo en el ejemplo del Titanic)
limpios   = (linea.strip() for linea in fichero)
campos    = (linea.split(',') for linea in limpios)
filtrados = (c for c in campos if c[0] == 'OK')
```

</div>

In [ ]:
# Comprehension List
squares_cl = [num ** 2 for num in range(5)]
print(squares_cl)
print(type(squares_cl))

- Generator Expressions

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 3</b>

Tienes esta lista:

```python
numeros = list(range(1, 11))   # [1, 2, ..., 10]
```

1. Crea una **list comprehension** `cuad_lista` con los cuadrados de cada numero. Imprime su tipo y la lista entera.
2. Crea una **generator expression** `cuad_gen` con los cuadrados. Imprime su tipo y el resultado de `print(cuad_gen)` (vas a ver `<generator object ...>`).
3. Usa `sum(...)` con un generator expression directamente para calcular la **suma de los cuadrados** de los numeros pares de la lista. (Ojo, los pares: 2, 4, 6, 8, 10).

> Resultado esperado: 4 + 16 + 36 + 64 + 100 = 220.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
numeros = list(range(1, 11))



In [ ]:
# === Solucion sugerida ===
# numeros = list(range(1, 11))
#
# # 1) List comprehension: crea TODA la lista en memoria
# cuad_lista = [x**2 for x in numeros]
# print(type(cuad_lista))   # <class 'list'>
# print(cuad_lista)         # [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
#
# # 2) Generator expression: solo describe COMO calcular, no calcula
# cuad_gen = (x**2 for x in numeros)
# print(type(cuad_gen))     # <class 'generator'>
# print(cuad_gen)           # <generator object ...>
#
# # 3) Suma de cuadrados de pares con generator expression
# # No hace falta corchetes adentro: sum() acepta el generator directo
# total = sum(x**2 for x in numeros if x % 2 == 0)
# print(total)              # 220


In [ ]:
# Cambia [] por ()
squares_ge = (num ** 2 for num in range(5))
print(squares_ge)

In [ ]:
next(squares_ge)

## Pipelines de Datos con Generadores

- Pipeline -> Generadores encadenados

- A partir de un fichero csv
  - Leer una línea
  - Dividir cada línea en una lista de valores
  - Extraer nombres de las columnas
  - Usar nombres de las columnas para crear un diccionario con los datos de cada línea
  - Filtrar los datos en base a algún criterio
  - Calcular algún valor para dichos criterios

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Pipeline de generadores: encadenar transformaciones</b>

Un **pipeline** es una serie de generadores donde **cada uno consume** los valores que produce el anterior. Ningun valor se almacena entero: fluye **paso a paso** por la cadena.

```
fichero -> generador1 (leer linea) -> generador2 (split) -> generador3 (filtrar) -> sum()
```

#### Por que esto es genial

| Ventaja | Detalle |
|---|---|
| **Memoria casi nula** | Solo una linea viva a la vez, da igual si el fichero tiene 100 GB. |
| **Composable** | Cada paso es una linea independiente, facil de leer y testear. |
| **Lazy** | Si solo te interesan las primeras 10 filas, el pipeline solo procesa esas. |
| **Reutilizable** | Cambiar un paso no afecta a los otros. |

#### Patron general

```python
# Paso 1: producir lineas crudas
lineas = (l for l in open('fichero.csv'))

# Paso 2: limpiar y dividir por columnas
campos = (l.rstrip().split(',') for l in lineas)

# Paso 3: tirar la cabecera
cabecera = next(campos)

# Paso 4: pasar a diccionarios
filas = (dict(zip(cabecera, c)) for c in campos)

# Paso 5: filtrar
adultos = (f for f in filas if int(f['edad']) >= 18)

# Paso 6: consumir
total = sum(1 for _ in adultos)
print(total)
```

> **No olvides cerrar el fichero** despues. Mejor todavia: usa `with open(...) as f:` y haz el pipeline dentro del bloque.

#### En IA esto es CRUCIAL

Los datasets de produccion **no caben en RAM**. Los generadores de Keras (`flow_from_directory`, `Sequence`) son exactamente este patron: producen lotes (`batches`) sin cargar nunca todo en memoria.

</div>

<div style="background-color:#f3f0ff;border-left:6px solid #7048e8;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#4c2d9e;font-size:14px;">PARA RECORDAR</b>

**El pipeline del Titanic que vas a ver:**

1. `lines = (line for line in file_handler)` -> producir lineas.
2. `list_line = (s.rstrip().split(',') for s in lines)` -> limpiar y dividir.
3. `cols = next(list_line)` -> extraer la cabecera (consume una fila).
4. `passengers = (dict(zip(cols, data)) for data in list_line)` -> pasar a dicts.
5. `passenger_in_class3_survived = (int(p['Survived']) for p in passengers if p['Pclass']=='3')` -> filtrar y proyectar.
6. `sum(passenger_in_class3_survived)` -> contar supervivientes.

Solo en el paso 6 se ejecuta TODO el pipeline, una linea a la vez. Lee el codigo con esta vision.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 4</b>

Sin abrir ningun fichero, vamos a probar un mini pipeline en memoria con esta lista de strings:

```python
datos = [
    'manzana,5',
    'pera,3',
    'uva,12',
    'kiwi,7',
    'mango,2'
]
```

Construye un pipeline de generadores que:

1. `divididos`: por cada linea, devuelva una lista `[fruta, cantidad_str]` (usa `split(',')`).
2. `parseados`: por cada elemento divido, devuelva una **tupla** `(fruta, int(cantidad))`.
3. `mayores`: filtre solo las frutas con cantidad **mayor o igual a 5**.
4. Imprime el **total de frutas** sumando con `sum(...)` sobre el generador `mayores` proyectado a la cantidad.

> Resultado esperado: 5 + 12 + 7 = 24.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
datos = [
    'manzana,5',
    'pera,3',
    'uva,12',
    'kiwi,7',
    'mango,2'
]



In [ ]:
# === Solucion sugerida ===
# datos = ['manzana,5', 'pera,3', 'uva,12', 'kiwi,7', 'mango,2']
#
# # Paso 1: dividir cada string por la coma -> ['manzana', '5'], ...
# divididos = (linea.split(',') for linea in datos)
#
# # Paso 2: convertir cantidad a int -> ('manzana', 5), ...
# parseados = ((fruta, int(cantidad)) for fruta, cantidad in divididos)
#
# # Paso 3: filtrar solo las que tienen cantidad >= 5
# mayores = ((fruta, cantidad) for fruta, cantidad in parseados if cantidad >= 5)
#
# # Paso 4: sumar las cantidades de las que pasan el filtro
# # NOTA: la suma se calcula recorriendo todo el pipeline UNA vez
# total = sum(cantidad for _, cantidad in mayores)
# print(total)   # 24


In [ ]:
from os import path
file_name = path.join("res", "titanic.csv")

In [ ]:
# GenExpr 01 -> Leer linea del fichero
file_handler = open(file_name)
lines = (line for line in file_handler)
print(lines)

In [ ]:
# GenExpr 02 -> Dividir cada linea
list_line = (s.rstrip().split(",") for s in lines)
print(list_line)

In [ ]:
cols = next(list_line)

In [ ]:
print(cols)

In [ ]:
# GenExpr 03 -> Diccionario, keys=nombre columna
passengers = (dict(zip(cols, data)) for data in list_line)
print(passengers)

In [ ]:
# GenExpr 04 -> Filtrado de datos
passenger_in_class3_survived = (int(passenger["Survived"]) for passenger in passengers if int(passenger["Pclass"]) == 3)
print(passenger_in_class3_survived)

In [ ]:
next(passenger_in_class3_survived)

In [ ]:
# Usa el Pipeline
passengers_in_class3_survived = sum(passenger_in_class3_survived)

In [ ]:
print(passengers_in_class3_survived)

In [ ]:
file_handler.close()

## Generadores para entrenar modelos

- Fichero de millones de líneas
- Mapeado lineal - y:f(x) = x

| x | y |
|---|---|
| 1 | 1 |
| 2 | 2 |
| 3 | 3 |
....

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Generadores para entrenar modelos: por que importan</b>

Cuando trabajas con datasets reales (millones de filas, GB o TB de imagenes) **no puedes cargar todo en RAM**. Esta es la situacion mas frecuente en proyectos de IA serios.

#### El truco: cargar por LOTES (batches)

Un **batch** es un subconjunto pequeno de datos (16, 32, 64 ejemplos). El modelo:

1. Lee un batch del disco.
2. Calcula la perdida.
3. Actualiza pesos.
4. Olvida ese batch y carga el siguiente.

Asi en cualquier instante solo hay 1 batch en memoria, no el dataset entero.

#### Quien suministra los batches?

Un **generador**. Por eso Keras acepta directamente un generador en `model.fit(generator, ...)` en lugar de un array NumPy.

```python
def generate_arrays_from_file(path, batchsize):
    while True:                          # bucle infinito (Keras lo controla)
        with open(path) as f:
            inputs, targets = [], []
            for linea in f:
                x, y = linea.strip().split(',')
                inputs.append([float(x)])
                targets.append([float(y)])
                if len(inputs) == batchsize:
                    yield np.array(inputs), np.array(targets)   # entrega un batch
                    inputs, targets = [], []                     # reinicia
```

#### Parametros importantes en `model.fit(...)`

| Parametro | Que es |
|---|---|
| `steps_per_epoch` | cuantos batches forman una epoca (= total / batch_size) |
| `epochs` | cuantas pasadas completas al dataset |
| `batch_size` | cuantos ejemplos por batch (lo controla el generador) |

#### Ojo: el generador debe ser INFINITO

A diferencia de los pipelines de antes, este generador NO termina (`while True`). Keras se encarga de pararlo despues de `steps_per_epoch * epochs` batches.

</div>

In [ ]:
from os import path
lineal_map = path.join("res", "huge_lineal_map.csv")

# Numero de filas
num_rows = 5000000

In [ ]:
# Crea fichero de prueba
with open(lineal_map, "w") as f:
    for i in range(num_rows):
        f.write(f"{i},{i}\n")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import numpy as np

- Función para leer el fichero en batches

In [ ]:
batch_size = 250

In [ ]:
# Carga Datos
def generate_arrays_from_file(path, batchsize):
    inputs = []
    targets = []
    batchcount = 0
    while True:
        with open(path) as f:
            for line in f:
                x,y = line.split(',')
                inputs.append(x)
                targets.append(y)
                batchcount += 1
                if batchcount > batchsize:
                    X = np.array(inputs, dtype='float32')
                    Y = np.array(targets, dtype='float32')
                    yield (X, Y)
                    inputs = []
                    targets = []
                    batchcount = 0

In [ ]:
# Crea el modelo
model = Sequential()
model.add(Input(shape=(1,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='linear'))

In [ ]:
# Compila el modelo
model.compile(loss='mean_absolute_error',
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['mean_squared_error'])

In [ ]:
# Entrena el modelo
model.fit(generate_arrays_from_file(lineal_map, batch_size), 
            steps_per_epoch=int(num_rows / batch_size), epochs=2)

In [ ]:
model.predict(np.array([13504, 2323]))

## Generadores de datos de imágenes con Keras

- ImageDataGenerator

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; ImageDataGenerator: generadores de imagenes en Keras</b>

Para datasets de **imagenes** Keras incluye `ImageDataGenerator`, una clase que automaticamente:

1. Lee imagenes desde disco.
2. Las redimensiona.
3. Las normaliza (divide por 255 para llevar a [0, 1]).
4. Aplica **augmentations** (rotaciones, flips, zooms aleatorios).
5. Las entrega en batches al modelo.

#### Tres formas principales de alimentarlo

| Metodo | De donde lee | Cuando usar |
|---|---|---|
| `.flow_from_directory(carpeta)` | Carpeta organizada por clases (un subdir por clase) | Lo mas comun para clasificacion de imagenes |
| `.flow(x, y)` | Arrays NumPy en memoria | Datasets pequenos como MNIST |
| `.flow_from_dataframe(df, ...)` | DataFrame con paths y labels | Cuando tienes un CSV con metadatos |

#### Estructura tipica de carpeta para `flow_from_directory`

```
res/images/
    perro/
        foto1.jpg
        foto2.jpg
    gato/
        foto3.jpg
        foto4.jpg
```

`generator.class_indices` te devuelve `{'gato': 0, 'perro': 1}` (orden alfabetico).

#### Diferencia clave con un generador 'a mano'

`ImageDataGenerator` ya implementa `__iter__` y `__next__`, asi que se comporta como cualquier iterador Python. **Pero tambien es infinito**, justo como necesita Keras.

</div>

### flow_from_directory
- Alimentar el generador desde un directorio de imágenes
- Fotos de una misma categoría deben ubicarse en una carpeta con el nombre de la clase
- Tamaño de lotes
- Redimensión de imágenes

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from os import path
images_folder = path.join("res", "images")
n_images = 23

In [ ]:
# Apuntar al generador al directorio de imagenes
data_generator = ImageDataGenerator()
generator = data_generator.flow_from_directory(images_folder, batch_size=1, target_size=(300, 300))

In [ ]:
print(f"{generator.class_indices}")
indices_classes = {v: k for k, v in generator.class_indices.items()}

In [ ]:
for _ in range(n_images):
    images, labels = next(generator)
    plt.imshow((images[0].astype(int)))
    plt.show()

    print(f"Label = {indices_classes[np.argmax(labels[0])]} {labels[0]}")

- Usar generador como conjunto de datos para entrenamiento o para validación

In [ ]:
# model.fit(generator, epochs = epochs, validation_data = generator_val)

### flow

- Alimentar el generador desde arrays NumPy
- MNIST https://en.wikipedia.org/wiki/MNIST_database

In [ ]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPool2D, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Carga dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_test.shape)
x_train = np.expand_dims(x_train, axis=-1)  #to fit flow rank 4 demands (nimages, width, height, depth)
x_test = np.expand_dims(x_test, axis=-1)
print(x_train.shape)  

print(type(x_train))

In [ ]:
# Crear y compilar modelo
model = Sequential()
model.add(Input(shape=(28,28,1)))
model.add(Conv2D(32,(3,3),activation='relu'))
model.add(MaxPool2D((2,2)))
model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPool2D((2,2)))
model.add(Flatten())
model.add(Dense(512,activation='relu'))
model.add(Dense(10,activation='softmax'))
 
model.compile(loss='sparse_categorical_crossentropy', optimizer='Adam', metrics=['accuracy'])

In [ ]:
datagen = ImageDataGenerator(rescale=1/255.,validation_split=0.2)

In [ ]:
my_batch_size = 64

In [ ]:
# Generadores para entrenamiento y validacion
training_generator = datagen.flow(x_train, y_train, batch_size=my_batch_size,subset='training',seed=7)
validation_generator = datagen.flow(x_train, y_train, batch_size=my_batch_size,subset='validation',seed=7)

In [ ]:
# Visualiza imagenes del generador
plt.figure(figsize=(10,5))
for i in range(6):
    plt.subplot(2,3,i+1)
    for x,y in training_generator:
        plt.imshow((x[0]/255).reshape(28,28),cmap='gray')
        plt.title(f'y={y[0]}')
        plt.axis('off')
        break
plt.tight_layout()
plt.show()

In [ ]:
# Entrena el modelo
model.fit(training_generator, steps_per_epoch=int((len(x_train)*0.8)//my_batch_size), epochs=4, 
          validation_data=validation_generator, validation_steps=int((len(x_train)*0.2)//my_batch_size))

In [ ]:
# Verificar precision del entrenamiento
score = model.evaluate(x_test, y_test, verbose=0)
print(f"Accuracy={score[1]}")

### Augmentations
- Transformaciones, variaciones de imágenes
- Data Augmentation
- https://www.pexels.com/photo/brown-animal-1454786/

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Data Augmentation: por que es decisivo en vision</b>

**Augmentation** = generar **variaciones aleatorias** de cada imagen sobre la marcha.

#### El problema

Si tu dataset tiene 1000 fotos de gatos, el modelo puede memorizarlas. Pero si **cada epoca** ve esas mismas 1000 fotos un poco rotadas, recortadas o invertidas... aprende a reconocer **gatos** en general, no esas 1000 fotos en particular.

| Sin augmentation | Con augmentation |
|---|---|
| Modelo ve 1000 fotos | Modelo ve 1000 fotos x 50 epocas = 50000 variaciones distintas |
| Tiende a sobreajustar (overfitting) | Generaliza mejor a fotos nuevas |
| Necesitas mas datos reales | Aprovechas mas los pocos que tienes |

#### Transformaciones tipicas

| Parametro | Que hace |
|---|---|
| `rotation_range=45` | rota +/- 45 grados |
| `width_shift_range=0.2` | desplaza horizontal hasta 20% |
| `height_shift_range=0.2` | desplaza vertical |
| `zoom_range=0.3` | zoom in/out |
| `horizontal_flip=True` | espeja izq-dcha |
| `vertical_flip=True` | espeja arriba-abajo |
| `brightness_range=(0.5, 1.5)` | varia brillo |
| `shear_range=0.2` | deformacion en angulo |
| `rescale=1./255` | normaliza a [0, 1] |

#### Cuidado: no toda transformacion es valida

- **`vertical_flip=True` en perros / gatos**: los animales no se ven volteados al reves en la realidad. Puede confundir al modelo.
- **`vertical_flip=True` en imagenes de satelite**: si funciona, no hay un 'arriba' fijo.
- **Rotacion grande en digitos MNIST**: un 6 rotado 180 grados parece un 9. Cuidado con cambiar la clase real.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 5</b>

**Reflexion (sin codigo).** Imagina que entrenas un clasificador de **placas de matricula**. Para cada uno de estos augmentations decide si SI o NO los aplicarias y por que:

1. `horizontal_flip=True`
2. `rotation_range=10`
3. `rotation_range=180`
4. `brightness_range=(0.5, 1.5)`
5. `vertical_flip=True`
6. `zoom_range=0.1`

Escribe la respuesta como comentarios en la celda.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu razonamiento como comentarios:

# 1) horizontal_flip=True ->
# 2) rotation_range=10 ->
# 3) rotation_range=180 ->
# 4) brightness_range=(0.5, 1.5) ->
# 5) vertical_flip=True ->
# 6) zoom_range=0.1 ->


In [ ]:
# === Solucion sugerida (criterio razonado) ===
#
# 1) horizontal_flip=True -> NO. Una matricula espejada cambia las letras
#    (B se convierte en algo distinto). El modelo aprenderia patrones falsos.
#
# 2) rotation_range=10 -> SI. En la realidad las fotos pueden estar un
#    poco torcidas (camara mal alineada, coche en pendiente). +/- 10 grados
#    es razonable y la matricula sigue legible.
#
# 3) rotation_range=180 -> NO. Una matricula del reves no existe en la realidad
#    y el modelo veria datos imposibles. Solo confunde.
#
# 4) brightness_range=(0.5, 1.5) -> SI. Es excelente: simula condiciones
#    de luz reales (dia nublado, sombra, sol directo).
#
# 5) vertical_flip=True -> NO. Lo mismo que la rotacion 180. Una matricula
#    boca abajo no es algo realista.
#
# 6) zoom_range=0.1 -> SI. Las matriculas se ven a distintas distancias en
#    fotos reales. Un zoom moderado ayuda a generalizar.
#
# REGLA GENERAL: solo aplica augmentations que generen imagenes que
# realmente podrian aparecer en produccion.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, ImageDataGenerator, img_to_array
import tensorflow.keras
from PIL import Image, ImageFilter

In [ ]:
from os import path
meerkat_path = path.join("res", "meerkat.jpg")

In [ ]:
# Carga la imagen, convierte a ndarray y expande las dimensiones
img = load_img(meerkat_path)
img = img_to_array(img)

print(img.shape)
img = np.expand_dims(img, axis=0)
print(img.shape)

In [ ]:
# Visualizar muestras de augmentations
def show_augmentations(generator):
    plt.figure(figsize=(5,5))
    for i in range(6):
        plt.subplot(2,3,i+1)
        for x in generator:
            plt.imshow(x[0]/255.)
            plt.xticks([])
            plt.yticks([])
            break
    plt.tight_layout()
    plt.show()

| Método | Descripción | Valor Muestra |
|--------|-------------|---------------|
| rotation_range | Rotación | 45 |
| width_shift_range | Desplazamiento horizontal | 0.3 |
| height_shift_range | Desplazamiento vertical | 0.3 |
| brightness_range | Rango de brillo | [0.1, 0.9] |
| rescale | Normaliza valores de pixels a un rango | 1/255. |
| shear_range | Shear | 45 |
| zoom_range | Zoom | 0.5 |
| channel_shift_range | Canales | 100 |
| horizontal_flip | Espejo en horizontal | True |
| vertical_flip | Espejo en vertical | True |
| preprocessing_function | Aplica función | f() |

In [ ]:
# Crear un generador con el metodo de augmentation
# image_gen = ImageDataGenerator()
# image_gen = ImageDataGenerator(rotation_range = 45)                   # rotation
# image_gen = ImageDataGenerator(height_shift_range = 0.2)              # desplazamiento horizontal
# image_gen = ImageDataGenerator(brightness_range = [0.1, 0.9])         # brillo
# image_gen = ImageDataGenerator(rescale = 1/255.)                      # normaliza valores de pixel (0,255) a (0,1)
# image_gen = ImageDataGenerator(shear_range = 40)                      # shear
# image_gen = ImageDataGenerator(zoom_range = 0.5)                      # zoom
# image_gen = ImageDataGenerator(channel_shift_range = 255)             # canales de color
image_gen = ImageDataGenerator(vertical_flip = True)                    # flip

generator = image_gen.flow(img, batch_size=1)
show_augmentations(generator)

In [ ]:
def blur(img_data):
    img = Image.fromarray(np.uint8(img_data))
    blurred_img = img.filter(ImageFilter.BLUR)
    blurred_data = np.array(blurred_img)
    return blurred_data
    
image_gen = ImageDataGenerator(preprocessing_function = blur)
generator = image_gen.flow(img, batch_size=1)
show_augmentations(generator)

## Generadores de datos custom con Keras

- A medida
- Sequence, clase base
https://www.tensorflow.org/api_docs/python/tf/keras/utils/Sequence

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Generador custom: heredar de <code>tf.keras.utils.Sequence</code></b>

Cuando los `ImageDataGenerator` no son suficientes (datos no son imagenes, formato raro, augmentations especiales) puedes crear tu **propio generador** heredando de `Sequence`.

#### Por que Sequence y no un generador normal con yield?

| Ventaja de `Sequence` | Detalle |
|---|---|
| **Multiprocessing seguro** | Keras puede paralelizar varios workers leyendo datos. |
| **Conoce su tamano** | Implementas `__len__()` y Keras sabe cuantos batches hay. |
| **Indexable** | `gen[i]` te da el batch i, en cualquier orden. |
| **Hook de fin de epoca** | Sobrescribes `on_epoch_end()` para barajar, recargar, etc. |

#### Metodos que DEBES implementar

```python
import tensorflow as tf
import numpy as np

class MiGen(tf.keras.utils.Sequence):

    def __init__(self, data, batch_size):
        self.data = data
        self.batch_size = batch_size

    def __len__(self):
        # Cuantos batches caben en una epoca
        return len(self.data) // self.batch_size

    def __getitem__(self, idx):
        # Devuelve el batch numero idx como (X, y)
        inicio = idx * self.batch_size
        fin    = inicio + self.batch_size
        batch  = self.data[inicio:fin]
        X = np.array([item['x'] for item in batch])
        y = np.array([item['y'] for item in batch])
        return X, y

    def on_epoch_end(self):
        # Se llama al acabar cada epoca: util para barajar
        np.random.shuffle(self.data)
```

#### Cuando usar Sequence vs ImageDataGenerator?

- `ImageDataGenerator`: solo imagenes en formato estandar. **Lo mas usado**.
- `Sequence`: cualquier otra cosa. Audio, texto, series temporales, datos tabulares enormes, formatos propietarios.

</div>

<div style="background-color:#f3f0ff;border-left:6px solid #7048e8;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#4c2d9e;font-size:14px;">PARA RECORDAR</b>

**Resumen del notebook 13: cuando uso cada cosa?**

| Caso | Herramienta |
|---|---|
| Calcular cuadrados al vuelo, pocos | `[x**2 for x in ...]` (lista) |
| Calcular cuadrados al vuelo, muchos | `(x**2 for x in ...)` (generator expr) |
| Funcion que produce valores con logica | `def f(): yield ...` (generator function) |
| Pipeline de transformaciones lazy | encadenar generator expressions |
| Datos que no caben en RAM | generador con `yield` (ej. lectura de fichero) |
| Imagenes para CNN | `ImageDataGenerator` + `flow_from_directory` |
| Pequeno dataset en memoria + augment | `ImageDataGenerator` + `.flow(x, y)` |
| Formato custom (audio, tabular pesado) | `class MiGen(tf.keras.utils.Sequence)` |

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 6</b>

**Ejercicio final integrador.** Vamos a recrear, en miniatura, lo que hace `Sequence` pero con un **generador con yield** simple.

Tienes:

```python
import numpy as np
np.random.seed(0)
X = np.arange(20).reshape(20, 1)        # 20 ejemplos de entrada
y = (X.flatten() * 2 + 3)                # y = 2*x + 3
batch_size = 4
```

1. Define una **funcion generadora infinita** `batch_gen(X, y, batch_size)` que:
   - Recorra los datos en orden.
   - Cuando se acabe un epoch, **vuelva a empezar** desde el principio (`while True`).
   - En cada llamada haga `yield` de una tupla `(X_batch, y_batch)` con tamano `batch_size`.

2. Crea el generador.
3. Llama a `next(...)` 6 veces (recuerda: hay 5 batches en una epoca, asi que la 6 ya esta en la segunda epoca) e imprime el `X_batch` de cada uno.

> Tip: usa un indice `i` que empiece en 0 y se incremente en `batch_size`, reseteandose cuando supere `len(X)`.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:
import numpy as np
np.random.seed(0)
X = np.arange(20).reshape(20, 1)
y = (X.flatten() * 2 + 3)
batch_size = 4



In [ ]:
# === Solucion sugerida ===
# import numpy as np
# np.random.seed(0)
# X = np.arange(20).reshape(20, 1)   # forma (20, 1)
# y = (X.flatten() * 2 + 3)            # y = 2x + 3, shape (20,)
# batch_size = 4
#
# def batch_gen(X, y, batch_size):
#     # Generador INFINITO: imprescindible para usarlo en model.fit con steps_per_epoch
#     i = 0
#     while True:
#         # si nos pasamos del final, reiniciamos al principio (nuevo epoch)
#         if i + batch_size > len(X):
#             i = 0
#         # extraemos un batch
#         X_batch = X[i:i + batch_size]
#         y_batch = y[i:i + batch_size]
#         # avanzamos el indice para la siguiente llamada
#         i += batch_size
#         # entregamos el batch y pausamos hasta el proximo next()
#         yield X_batch, y_batch
#
# # Crear el generador
# gen = batch_gen(X, y, batch_size)
#
# # Pedir 6 batches: vemos el wrap-around al volver al principio
# for paso in range(6):
#     Xb, yb = next(gen)
#     print(f'paso {paso} -> X_batch.flatten() = {Xb.flatten()}')


In [ ]:
from tensorflow.keras.models import Sequential
import tensorflow as tf

In [ ]:
# Clase debe sobreescribir algunos metodos 
class CustomDataGenerator(tf.keras.utils.Sequence):
    
    # Inicializacion
    def __init__(self, data, batch_size, *args, **kwargs):
        self.data = data
        self.batch_size = batch_size
        pass

    # Devuelve el numero de batches
    def __len__(self):    
        return total_data / self.batch_size

    # Devuelve un batch de datos
    def __getitem__(self, index):
        return X, y

    # Metodo opcional para ejecutar algo al final de cada epoch, (mezclar datos)
    def on_epoch_end(self):
        pass

In [ ]:
class CustomDataGenerator(tf.keras.utils.Sequence):
    
    # Inicializacion
    def __init__(self, data, x_col, y_col=None, batch_size=32, num_classes=None, shuffle=True):
        self.batch_size = batch_size
        self.data = data
        self.indices = self.data.index.tolist()
        self.num_classes = num_classes
        self.shuffle = shuffle
        self.x_col = x_col
        self.y_col = y_col
        self.on_epoch_end()

    # Devuelve el numero de batches
    def __len__(self):
        return len(self.indices) // self.batch_size

    # Devuelve un batch de datos
    def __getitem__(self, index):
        # Genera indices del batch
        index = self.index[index * self.batch_size:(index + 1) * self.batch_size]
        # Encuentra la lista de ID's
        batch = [self.indices[k] for k in index]
        
        # Genera los datos
        X, y = self.__get_data(batch)
        return X, y

    # Metodo opcional para ejecutar algo al final de cada epoch, (mezclar datos)
    def on_epoch_end(self):
        self.index = np.arange(len(self.indices))
        if self.shuffle == True:
            np.random.shuffle(self.index)

    # Metodo privado para la generacion de datos
    def __get_data(self, batch):
        X = []  # logica para cargar los datos de ficheros
        y = []  # logica para las variables target
        
        # Generar los datos
        for i, id in enumerate(batch):
            # Guardar un sample
            X[i,] = # logica para generar un sample (preprocesar/aumentar)
            y[i] = # etiquetas

        return X, y      

In [ ]:
# Usar un custom data set generator
train_generator = CustomDataGenerator(dataframe, batch_size = 32, num_classes = 2, shuffle = True)
validation_generator = CustomDataGenerator(dataframe, batch_size = 32, num_classes = 2, shuffle = True)

model.fit(train_generator, epochs, validation_data=validation_generator)